# Import Required Libraries
Import the necessary libraries, including NumPy, Pandas, Matplotlib, Seaborn, TensorFlow, and others.

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import pymongo
import os
from tensorflow.keras.utils import Sequence, to_categorical
import gc  # Garbage Collector
from tqdm import tqdm  # Progress bar

# Load and Merge Data
Load data from MongoDB, convert to DataFrames, and merge them.

In [ ]:
# Load and Merge Data

# Function to load data from MongoDB
def load_data():
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    # Load data
    canvassamples = list(db['canvassamples'].find())
    fingerprints = list(db['fingerprints'].find())
    
    # Convert to DataFrames
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    # Merge DataFrames
    merged_df = pd.merge(canvassamples_df, fingerprints_df, 
                         left_on='fingerprintId', 
                         right_on='_id', 
                         suffixes=('_sample', '_fingerprint'))
    
    return merged_df

# Load data
merged_df = load_data()
print(f"Gesamtdatensatz enthält {len(merged_df)} Einträge.")

# Preprocess Data
Process images and prepare them for model training.

In [ ]:
# Preprocess Data

# Function to process images in RGB
def process_image_rgb(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')  # Convert image to RGB
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0  # Normalize to values between 0 and 1
        return image_array
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

# Function to extract images and labels from DataFrame
def extract_images_from_df(df, example_user_id):
    images = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Lade Bilder"):
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Example user ID
example_user_id = 'benutzername_1'

# Create DataFrames for each user
user_ids = merged_df['username'].unique()
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Sample data for the example user
user_df = user_dfs[example_user_id].sample(n=12000, random_state=42)

# Add negative examples and limit
negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
negative_df = negative_df.sample(n=12000, random_state=42)

# Split into Train/Val/Test sets (70% Train, 20% Val, 10% Test)
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

# Process image data
try:
    X_train, y_train = extract_images_from_df(train_df, example_user_id)
    X_val, y_val = extract_images_from_df(val_df, example_user_id)
    X_test, y_test = extract_images_from_df(test_df, example_user_id)
except Exception as e:
    print(f"Fehler bei der Bilddatenverarbeitung: {e}")

# Free up memory
gc.collect()

# Split Data into Train, Validation, and Test Sets
Split the data into training, validation, and test sets.

In [4]:
# Split Data into Train, Validation, and Test Sets

# Split into Train/Val/Test sets (70% Train, 20% Val, 10% Test)
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

# Process image data
try:
    X_train, y_train = extract_images_from_df(train_df, example_user_id)
    X_val, y_val = extract_images_from_df(val_df, example_user_id)
    X_test, y_test = extract_images_from_df(test_df, example_user_id)
except Exception as e:
    print(f"Fehler bei der Bilddatenverarbeitung: {e}")

# Free up memory
gc.collect()

# Define CNN Model
Define a Convolutional Neural Network (CNN) model using TensorFlow/Keras.

In [5]:
# Neues Modell mit prozentualen Vorhersagen erstellen
def create_model_with_probabilities(input_shape):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))  # Sigmoid für prozentuale Vorhersagen
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [6]:
# Define CNN Model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

# Train the Model
Train the CNN model on the training data and validate it on the validation data.

In [ ]:
# Train the Model

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [8]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Beispielhafte Eingabeform
input_shape = (224, 224, 3)  # Passen Sie dies an Ihre Daten an
model_with_probabilities = create_model_with_probabilities(input_shape)

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history_prob = model_with_probabilities.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    batch_size=1  # Passe die Batch-Größe an
)

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.plot(history_prob.history['accuracy'])
plt.plot(history_prob.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(12, 6))
plt.plot(history_prob.history['loss'])
plt.plot(history_prob.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Evaluate the Model
Evaluate the model on the test data and print the results.

In [ ]:
# Evaluate the Model

# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions on the test data
y_pred = model.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype("int32").flatten()

# Print classification report
print(classification_report(y_test, y_pred_classes))

# Plot confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# Make Predictions and Analyze Results
Make predictions on the test data, generate a classification report, confusion matrix, and ROC curve.

In [ ]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions on the test data
y_pred = model.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype("int32").flatten()

# Print classification report
print(classification_report(y_test, y_pred_classes))

# Plot confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Assuming that the model is already trained and named 'model'
# If not, please make sure to train the model before running this code

# Function to process and prepare images from the dataframe
def prepare_images(df, num_samples_per_user):
    images = []
    labels = []
    user_sampled_dfs = []
    for user_id in user_ids:
        user_df = df[df['username'] == user_id].sample(n=num_samples_per_user, random_state=42)
        user_sampled_dfs.append(user_df)
        for _, row in user_df.iterrows():
            image = process_image_rgb(row['sampleData'])
            if image is not None:
                images.append(image)
                labels.append(user_id)
    return np.array(images), np.array(labels)

# Function to plot prediction results
def plot_prediction_results(results_df, num_samples, example_user_id):
    plt.figure(figsize=(15, 10))
    correct_predictions = results_df[(results_df['Actual User'] == example_user_id) & (results_df['Predicted User'] == example_user_id)]
    incorrect_predictions = results_df[(results_df['Actual User'] != example_user_id) & (results_df['Predicted User'] == example_user_id)]
    other_correct_predictions = results_df[(results_df['Actual User'] != example_user_id) & (results_df['Predicted User'] != example_user_id)]
    
    plt.scatter(correct_predictions.index, correct_predictions['Actual User'], color='green', label='Correct Predictions (User)')
    plt.scatter(incorrect_predictions.index, incorrect_predictions['Actual User'], color='red', label='Incorrect Predictions (User)')
    plt.scatter(other_correct_predictions.index, other_correct_predictions['Actual User'], color='blue', label='Correct Predictions (Other)')
    
    plt.xlabel('Sample Index')
    plt.ylabel('User')
    plt.title(f'Prediction Results for {num_samples} Samples per User')
    plt.legend()
    plt.show()

# Function to plot pie chart for prediction accuracy
def plot_pie_chart(results_df, example_user_id):
    total_samples = len(results_df)
    correct_predictions = len(results_df[(results_df['Actual User'] == example_user_id) & (results_df['Predicted User'] == example_user_id)])
    incorrect_predictions = len(results_df[(results_df['Actual User'] != example_user_id) & (results_df['Predicted User'] == example_user_id)])
    other_correct_predictions = len(results_df[(results_df['Actual User'] != example_user_id) & (results_df['Predicted User'] != example_user_id)])
    other_incorrect_predictions = len(results_df[(results_df['Actual User'] == example_user_id) & (results_df['Predicted User'] != example_user_id)])
    
    labels = ['Correct Predictions (User)', 'Incorrect Predictions (User)', 'Correct Predictions (Other)', 'Incorrect Predictions (Other)']
    sizes = [correct_predictions, incorrect_predictions, other_correct_predictions, other_incorrect_predictions]
    colors = ['green', 'red', 'blue', 'orange']
    
    plt.figure(figsize=(8, 8))
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
    plt.title('Prediction Accuracy Distribution')
    plt.show()

# Test with 3 samples from each user
num_samples = 3
X_test_users, y_test_users = prepare_images(merged_df, num_samples)

# Get predictions
y_pred = model.predict(X_test_users)
y_pred_labels = [example_user_id if pred > 0.5 else 'other_user' for pred in y_pred.flatten()]

# Create a DataFrame for visualization
results_df = pd.DataFrame({
    'Actual User': y_test_users,
    'Predicted User': y_pred_labels
})

# Plotting the results
plot_prediction_results(results_df, num_samples, example_user_id)

# Confusion Matrix
conf_matrix = confusion_matrix(results_df['Actual User'] == example_user_id, results_df['Predicted User'] == example_user_id)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Other', example_user_id], yticklabels=['Other', example_user_id])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix with {num_samples} Samples per User')
plt.show()

# Accuracy
accuracy = accuracy_score(results_df['Actual User'] == example_user_id, results_df['Predicted User'] == example_user_id)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Classification Report
print(classification_report(results_df['Actual User'] == example_user_id, results_df['Predicted User'] == example_user_id))

# Pie Chart for Prediction Distribution
prediction_counts = results_df['Predicted User'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(prediction_counts, labels=prediction_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Prediction Distribution')
plt.show()

# Test with 1000 samples from each user
num_samples = 100
X_test_users_large, y_test_users_large = prepare_images(merged_df, num_samples)

# Get predictions
y_pred_large = model.predict(X_test_users_large)
y_pred_labels_large = [example_user_id if pred > 0.5 else 'other_user' for pred in y_pred_large.flatten()]

# Create a DataFrame for visualization
results_df_large = pd.DataFrame({
    'Actual User': y_test_users_large,
    'Predicted User': y_pred_labels_large
})

# Plotting the results
plot_prediction_results(results_df_large, num_samples, example_user_id)

# Confusion Matrix
conf_matrix_large = confusion_matrix(results_df_large['Actual User'] == example_user_id, results_df_large['Predicted User'] == example_user_id)
sns.heatmap(conf_matrix_large, annot=True, fmt='d', cmap='Blues', xticklabels=['Other', example_user_id], yticklabels=['Other', example_user_id])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix with {num_samples} Samples per User')
plt.show()

# Accuracy
accuracy_large = accuracy_score(results_df_large['Actual User'] == example_user_id, results_df_large['Predicted User'] == example_user_id)
print(f'Accuracy: {accuracy_large * 100:.2f}%')

# Classification Report
print(classification_report(results_df_large['Actual User'] == example_user_id, results_df_large['Predicted User'] == example_user_id))

# Pie Chart for Prediction Accuracy
plot_pie_chart(results_df_large, example_user_id)

In [ ]:
import random
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming that the model is already trained and named 'model'
# If not, please make sure to train the model before running this code

# Function to process and prepare images from the dataframe
def prepare_images(df, num_samples_per_user):
    images = []
    labels = []
    user_sampled_dfs = []
    for user_id in user_ids:
        user_df = df[df['username'] == user_id].sample(n=num_samples_per_user, random_state=42)
        user_sampled_dfs.append(user_df)
        for _, row in user_df.iterrows():
            image = process_image_rgb(row['sampleData'])
            if image is not None:
                images.append(image)
                labels.append(user_id)
    return np.array(images), np.array(labels)

def prepare_images(df, total_samples):
    users = df['username'].unique()
    samples_per_user = total_samples // len(users)
    remaining_samples = total_samples % len(users)
    
    X = []
    y = []
    
    for user in users:
        user_samples = df[df['username'] == user]
        num_samples = samples_per_user + (1 if remaining_samples > 0 else 0)
        remaining_samples -= 1
        
        sampled_data = user_samples.sample(n=num_samples, random_state=42)
        X.extend(sampled_data['sampleData'].tolist())
        y.extend([user] * num_samples)
    
    return X, y

# Funktion zur Evaluierung des Modells mit prozentualen Vorhersagen
def evaluate_model_with_probabilities(model, X_test, y_test, example_user_id):
    y_pred_prob = model.predict(X_test).flatten()
    y_pred_labels = [example_user_id if prob > 0.7 else 'other_user' for prob in y_pred_prob]
    
    results_df = pd.DataFrame({
        'Actual User': y_test,
        'Predicted User': y_pred_labels,
        'Prediction Probability': y_pred_prob
    })
    
    return results_df

# Test with few samples from each user
num_samples = 3
X_test_users, y_test_users = prepare_images(merged_df, num_samples)

# Evaluierung des neuen Modells
results_df_prob = evaluate_model_with_probabilities(model_with_probabilities, X_test_users, y_test_users, example_user_id)

# Plotting the results
plot_prediction_results(results_df_prob, num_samples, example_user_id)

# Confusion Matrix
conf_matrix_prob = confusion_matrix(results_df_prob['Actual User'] == example_user_id, results_df_prob['Predicted User'] == example_user_id)
sns.heatmap(conf_matrix_prob, annot=True, fmt='d', cmap='Blues', xticklabels=['Other', example_user_id], yticklabels=['Other', example_user_id])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix with {num_samples} Samples per User (Probabilities)')
plt.show()

# Accuracy
accuracy_prob = accuracy_score(results_df_prob['Actual User'] == example_user_id, results_df_prob['Predicted User'] == example_user_id)
print(f'Accuracy: {accuracy_prob * 100:.2f}%')

# Classification Report
print(classification_report(results_df_prob['Actual User'] == example_user_id, results_df_prob['Predicted User'] == example_user_id))

# Pie Chart for Prediction Accuracy
plot_pie_chart(results_df_prob, example_user_id)

# Test with 10,000 samples in total from different users
total_samples = 10000
X_test_users_large, y_test_users_large = prepare_images(merged_df, total_samples)

# Evaluierung des neuen Modells
results_df_prob_large = evaluate_model_with_probabilities(model_with_probabilities, X_test_users_large, y_test_users_large, example_user_id)

# Plotting the results
plot_prediction_results(results_df_prob_large, total_samples, example_user_id)

# Confusion Matrix
conf_matrix_prob_large = confusion_matrix(results_df_prob_large['Actual User'] == example_user_id, results_df_prob_large['Predicted User'] == example_user_id)
sns.heatmap(conf_matrix_prob_large, annot=True, fmt='d', cmap='Blues', xticklabels=['Other', example_user_id], yticklabels=['Other', example_user_id])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix with {total_samples} Samples in Total (Probabilities)')
plt.show()

# Accuracy
accuracy_prob_large = accuracy_score(results_df_prob_large['Actual User'] == example_user_id, results_df_prob_large['Predicted User'] == example_user_id)
print(f'Accuracy: {accuracy_prob_large * 100:.2f}%')

# Classification Report
print(classification_report(results_df_prob_large['Actual User'] == example_user_id, results_df_prob_large['Predicted User'] == example_user_id))

# Pie Chart for Prediction Accuracy
plot_pie_chart(results_df_prob_large, example_user_id)

# Vergleich der Performance beider Modelle
print("Vergleich der Performance beider Modelle:")
print(f"Altes Modell - Accuracy: {accuracy * 100:.2f}%")
print(f"Neues Modell - Accuracy (few samples): {accuracy_prob * 100:.2f}%")
print(f"Neues Modell - Accuracy (10,000 samples): {accuracy_prob_large * 100:.2f}%")

In [ ]:
import tensorflow as tf

# Funktion zum Freigeben des GPU-Speichers
def release_gpu_memory():
    # TensorFlow GPU-Speicher freigeben
    tf.keras.backend.clear_session()

# Beispielaufruf der Funktion
release_gpu_memory()